# Week 4: Gene Set Enrichment Analysis
In the previous module, we performed Differential Gene Expression (DGE) analysis to identify individual genes that were significantly up- or down-regulated after treating airway smooth muscle cells with Dexamethasone. This gave us a powerful, gene-by-gene view of the cellular response.

While a list of significant genes is a great start, it doesn't always tell the whole story. Biological functions are rarely the result of a single gene; they are the product of complex, interacting networks of genes. This is where **Gene Set Enrichment Analysis (GSEA)** comes in.
GSEA extends our DGE analysis by asking a different question: *Are the genes within a known biological pathway or process collectively showing a significant change in our experiment?* Instead of focusing on individual trees, GSEA helps us see the entire forest. It connects our expression data to a massive knowledge base of predefined "gene sets," which represent pathways, disease signatures, or cellular states.

In this module, we will use the DGE results from last week to perform GSEA and uncover the broader biological themes at play in our `'Dexamethasone'` vs. `'Untreated'` comparison.

## Learning objectives

By the end of this module, you will be equipped to accomplish the following tasks:
- Explain the conceptual framework behind Gene Set Enrichment Analysis.
- Prepare a ranked gene list from DGE results for GSEA.
- Perform GSEA using the Python library `gseapy`.
- Interpret key GSEA metrics, including the Normalized Enrichment Score (NES) and False Discovery Rate (FDR).
- Visualize and draw meaningful biological conclusions from GSEA results.

**Source:** Subramanian A, Tamayo P, Mootha VK, Mukherjee S, Ebert BL, Gillette MA, Paulovich A, Pomeroy SL, Golub TR, Lander ES, Mesirov JP. Gene set enrichment analysis: a knowledge-based approach for interpreting genome-wide expression profiles. Proc Natl Acad Sci U S A. 2005 Oct 25;102(43):15545-50. doi: <a href=https://doi.org/10.1073/pnas.0506580102>10.1073/pnas.0506580102</a>. Epub 2005 Sep 30. PMID: 16199517; PMCID: PMC1239896.

Let's start by installing the required libraries and importing them.

**Run the code below.**

In [ ]:
!pip -q install gseapy

In [ ]:
# Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gseapy as gp
from gseapy import dotplot, barplot

In [ ]:
# This cell contains a helper function we'll use in this module.
# You don't need to understand this code, but make sure to run the cell!
def plot_volcano(dge_df, p_threshold=0.05, log2fc_threshold=1, top_N=10):
    plt.figure(figsize=(12, 8))

    # Plot non-significant genes in gray
    plt.scatter(dge_df['log2FoldChange'], -np.log10(dge_df['padj']), edgecolors='lightgray', facecolors='none', label='Non-significant', alpha=0.7)

    # Plot upregulated genes in organge
    plt.scatter(dge_df[(dge_df['log2FoldChange'] > log2fc_threshold) & (dge_df['padj'] < p_threshold)]['log2FoldChange'],
                -np.log10(dge_df[(dge_df['log2FoldChange'] > log2fc_threshold) & (dge_df['padj'] < p_threshold)]['padj']),
                edgecolors='orange', facecolors='none', label='Upregulated', alpha=0.7)
    
    # Plot downregulated genes in blue
    plt.scatter(dge_df[(dge_df['log2FoldChange'] < -log2fc_threshold) & (dge_df['padj'] < p_threshold)]['log2FoldChange'],
                -np.log10(dge_df[(dge_df['log2FoldChange'] < -log2fc_threshold) & (dge_df['padj'] < p_threshold)]['padj']),
                edgecolors='skyblue', facecolors='none', label='Downregulated', alpha=0.7)

    # Plotting and labeling most significant genes with fill
    if top_N > 0:
        top_genes = dge_df.sort_values('padj', ascending=True).head(top_N)
        for _, row in top_genes.iterrows():
            color = 'orange' if row['log2FoldChange'] > 0 else 'skyblue'
            plt.scatter(row['log2FoldChange'], -np.log10(row['padj']), edgecolors=color, facecolors=color, s=50)
            plt.text(row['log2FoldChange'], -np.log10(row['padj']), row['symbol'], fontsize=9, ha='right')

    # Adding lines for thresholds
    sig_line = -np.log10(p_threshold)
    plt.axhline(y=sig_line, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(x=log2fc_threshold, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(x=-log2fc_threshold, color='black', linestyle='--', linewidth=0.5)

    # Adding details to plot
    plt.title('Volcano Plot of Differential Gene Expression')
    plt.xlabel('Log2 Fold Change')
    plt.ylabel('-Log10 Adjusted P-Value')
    plt.legend()
    plt.show()

## Preparing the Dataset

Instead of re-running the entire DGE analysis from scratch, we'll load a pre-computed results file. This file, `complete_DGE_comparisons_symbolic.csv`, contains the full DGE statistics for all treatment conditions in the Himes et. al study compared to the `'Untreated'` control.

This allows us to quickly access the results for our `'Dexamethasone'` comparison and sets us up to analyze the `'Albuterol'` treatment later in the graded questions.

In [ ]:
dge_df = pd.read_csv('data/complete_DGE_comparisons_symbolic.csv')
dge_df

## How Gene Set Enrichment Analysis Works

GSEA is a powerful method, but its core logic is elegant. It can be broken down into three main steps:
1. **Rank All Genes:** GSEA doesn't just look at the handful of genes that pass a strict significance threshold (like `padj < 0.05`). Instead, it considers all genes from the DGE analysis. We will rank every gene from most upregulated to most downregulated. A great metric for this is the `stat` value from our `DESeq2` results, as it combines the effect size (log2FoldChange) and the statistical confidence into a single "signal-to-noise" score.
2. **Test for Enrichment:** For a given gene set (e.g., "Hallmark Inflammatory Response"), GSEA walks down our ranked list. It calculates a running "Enrichment Score" (ES). Every time it encounters a gene that is in the gene set, the score increases. Every time it encounters a gene that is not in the set, the score decreases. The maximum deviation from zero in this "random walk" is the final ES. A high positive ES means the gene set is concentrated at the top of our list (upregulated), while a high negative ES means it's concentrated at the bottom (downregulated).
3. **Assess Significance:** How do we know if our ES is meaningful? GSEA determines this by permutation testing. It shuffles the gene labels in our ranked list thousands of times and re-calculates the ES for each shuffle. This creates a null distribution of random scores.
    - **Normalized Enrichment Score (NES):** Our actual ES is then compared to this null distribution and a new score is calculated, called the normalized enrichment score (NES). A high positive or negative NES tells us that our gene set is significantly more enriched than we'd expect by random chance.
    - **FDR q-value:** Just like the padj from last module, the False Discovery Rate (FDR) corrects for the fact that we are testing thousands of gene sets at once. It estimates the probability that a given gene set is a false positive. We will use the FDR to determine significance.

## Performing GSEA on Dexamethasone Treatment

Let's apply these concepts to our `'Dexamethasone'` vs. `'Untreated'` data.

First, we'll create a subset of our DGE results for just the `'Dexamethasone'` comparison.

In [ ]:
# Create a copy of dexamethosone comparison subset
dex_df = dge_df[dge_df['treatment_protocol'] == 'Dexamethasone'].copy()
dex_df

This is what our data looks like in a volcano plot.

In [ ]:
# Visualize significant genes in volcano plot
plot_volcano(dex_df)

Then, we will create our ranked gene list based on the `stat` column. 

- Note that GSEA libraries like `gseapy` require gene symbols, which is why we converted from Ensembl IDs last week.

In [ ]:
# First, we do some data cleaning: retrieve just the symbol gene IDs and stat scores,
# and remove rows with missing values and duplicated genes.
gene_stats = dex_df[['symbol', 'stat']].dropna()
gene_stats = gene_stats.drop_duplicates('symbol')

# Rank genes from highest to lowest stat scores
ranked_genes = gene_stats.sort_values(by='stat', ascending=False).reset_index(drop=True)
ranked_genes

Our `ranked_genes` DataFrame is now perfectly formatted for GSEA: a list of gene symbols, ranked from most upregulated (high positive `stat`) to most downregulated (high negative `stat`).

---
**Q*1: Inspect the `ranked_genes` DataFrame. What is the gene with the highest stat score (most upregulated)? What is the gene with the lowest stat score (most downregulated)? Use `.head(1)` and `.tail(1)` attributes to find out.**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Write your code here


---
## Choosing a Gene Set Library

Next, we need a collection of gene sets to test against. `gseapy` comes with preloaded gene sets available from the <a href=https://maayanlab.cloud/Enrichr/#libraries>Enrichr Library</a>. We can look at the collections of gene sets for specific species (e.g., `'Human'`, `'Mouse'`, `'Yeast'`, `'Fly'`, `'Fish'`, `'Worm'`) and choose an appropriate one. If we wanted to see all the available gene sets, we could look through the `human_gene_sets` list.

In [ ]:
# Shows gene sets available from Enrichr Library for a specified species (e.g., 'Human', 'Mouse', 'Yeast', 'Fly', 'Fish', 'Worm')
human_gene_sets = gp.get_library_name(organism='Human')
print(f"Number of human gene sets: {len(human_gene_sets)}")
print(f"First 5 human gene sets: {human_gene_sets[:5]}")

For this tutorial, we’ll use the Hallmark gene set collection (`MSigDB_Hallmark_2020`) from the Molecular Signatures Database (MSigDB). MSigDB houses widely used gene sets, and the Hallmark 2020 subset consists of 50 refined, biologically meaningful signatures representing key processes for us to give our data some context.

---
**Q*2: Find the `'MSigDB_Hallmark_2020'` library in the `human_gene_sets` list. Python lists have a handy `.index()` method.**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Answer:
msigdb_index = human_gene_sets.index()
print(f"{human_gene_sets[msigdb_index]} gene set is located at index {msigdb_index} in the Enrichr Library.")

---
We'll now use the `gseapy` `prerank()` function to perform GSEA.

| Function | Input parameters | Output | Syntax |
| --- | --- | --- | --- |
| `prerank()` | `rnk`, `gene_sets`, `seed`, ...| a table with detailed metrics such as raw enrichment scores (ES) for each gene set  | `prerank(rnk, gene_sets, seed, ...)` |

Parameters (not exhaustive):
- `rnk`: The pre-ranked gene list dataset, often as a `pandas` DataFrame.
- `gene_sets`: The gene set library to be used for enrichment analysis (e.g., KEGG, GO, or custom GMT files).
- `seed`: A numerical value that acts as an initialization point for the random number generator. By using the same `seed`, users can generate identical outputs, assuring reproducibility. We set `seed=0` here.

Output:
- It generates output, often in a tabular format (e.g., .txt, .csv), with detailed metrics for each analyzed gene set. We can visualize these metrics using the `plot()` function, which we will see later.

In [ ]:
# Run GSEA with our ranked genes and specified gene set from Enrichr Library.
# The 'prerank' function is used because we have a pre-ranked list of genes.
gsea_results = gp.prerank(
    rnk=ranked_genes, 
    gene_sets='MSigDB_Hallmark_2020',
    seed=0
)

# Get the results as a DataFrame and display it
gsea_results_df = gsea_results.res2d
gsea_results_df

The `gsea_results_df` DataFrame gives us the key metrics for each Hallmark gene set. Let's focus on:
- **Term:** The name of the gene set (e.g., Adipogenesis).
- **ES:** The raw Enrichment Score.
- **NES:** The Normalized Enrichment Score. 
    - Positive values indicate enrichment in the upregulated genes.
    - Negative values indicate enrichment in the downregulated genes.
- **FDR q-val:** The False Discovery Rate. We'll use a threshold of < 0.25 to identify significant pathways.

---
**Q*3 (Optional): This question should be somewhat challenging.
Try filtering the `gsea_results_df` DataFrame to keep only genes the most significant gene sets having `FDR q-val < 0.25` and sort these significant gene sets by the `'NES'` column in descending order. Based on this filtered DataFrame, which gene set has the highest NES? Which gene set has the lowest NES?**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Filter for significant results
significant_results = gsea_results_df[...]

# Sort the filtered DataFrame by NES
sorted_significant_results = significant_results.sort_values(by=..., ascending=False)

sorted_significant_results.head(1), sorted_significant_results.tail(1)

<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Visualizing and Interpreting Enrichment

To truly understand what the ES and NES mean, let's visualize the enrichment plots for a top upregulated and a top downregulated pathway.

**Upregulated Pathway: Adipogenesis**

Looking at our new GSEA results, the pathway with the most significant positive enrichment score is `'Adipogenesis'`. This is the biological process of cell differentiation that generates adipocytes (fat cells).

In [ ]:
# Plotting the enrichment score of the E2F targets
terms_to_plot = ['Adipogenesis']	
gsea_results.plot(terms=terms_to_plot)

gsea_results_df[gsea_results_df['Term'].isin(terms_to_plot)][["Term", "ES", "NES", "FDR q-val"]]

The plot shows:
- The running Enrichment Score (blue line) as it moves down the ranked list (from left to right, upregulated to downregulated).
- The "barcodes" (blue vertical lines) at the bottom indicate where the genes belonging to the `'Adipogenesis'` set appear in our list.

Notice how the blue line peaks on the left side of the plot. The cluster of blue barcode lines in this region indicates that genes involved in the cell cycle are disproportionately found at the top of our ranked list (i.e., they are associated with upregulation by Dexamethasone). The positive `NES` confirms this trend. While the `FDR q-value` is not highly significant (0.18), it still suggests a coordinated upregulation of these genes that is unlikely to be random. This aligns with the known complex roles of glucocorticoids like Dexamethasone, which can promote fat storage and influence metabolism.

**Downregulated Pathway: E2F Targets**

Now let's look at the pathway with the most significant negative enrichment score. One example would be `'E2F Targets'`. E2F transcription factors are regulators of the cell cycle, and their target genes are crucial for cell division and DNA replication.

In [ ]:
# Plotting the enrichment score of the adipogenesis pathway
terms_to_plot = ['E2F Targets']
gsea_results.plot(terms=terms_to_plot)

gsea_results_df[gsea_results_df['Term'].isin(terms_to_plot)][["Term", "ES", "NES", "FDR q-val"]]

Here, the pattern is the complete opposite. The enrichment score (blue line) reaches its minimum peak on the far right, and the barcode lines are densely clustered in the downregulated portion of our gene list. This provides strong evidence that genes promoting cell cycle progression are systematically and significantly suppressed in the Dexamethasone treated cells. This result makes perfect biological sense, since Dexamethasone is a potent anti-inflammatory and immunosuppressive drug known to inhibit the proliferation of many cell types, including smooth muscle cells.

## Summarizing GSEA Results

Looking at individual plots is insightful, but we need a high-level summary. A bar plot of the top enriched pathways is an excellent way to see the main biological stories at a glance. We'll plot the `'NES'` values for all significantly enriched pathways (`FDR q-value < 0.25`).

In [ ]:
# Filter for significant results
significant_results = gsea_results_df[gsea_results_df['FDR q-val'] < 0.25]

# Create barplot
ax = barplot(significant_results,
             column='NES',
             title='MSigDB Hallmark 2020 - Dexamethasone vs. Untreated',
             top_term=10,
             cmap='viridis_r',
             figsize=(6, 8))

This plot visualizes the top 10 most enriched pathways, sorted by their `NES`. A key observation is that all top 10 pathways have positive `NES` values. This indicates that the **upregulation of certain biological processes is the most dominant signal** in this GSEA analysis.

Specifically, we see a strong, coordinated upregulation of metabolic pathways like **Adipogenesis** (fat metabolism) and **Oxidative Phosphorylation** (cellular respiration). This aligns with the known role of glucocorticoids in profoundly altering cellular metabolism.

It is important to note that this plot does not mean there are no downregulated pathways. 

In [ ]:
# Order significant pathways by lowest NES to highest NES 
downregulated = significant_results.sort_values(by='NES', ascending=True)

# Top 10 downregulated pathways
downregulated[:10][["Term", "NES"]]

By inspecting the bottom 10 results in our table, we can confirm that pathways related to cell division, such as **E2F Targets**, are indeed significantly downregulated with strong negative NES scores. However, the upregulation of the metabolic pathways is statistically more prominent, which is why they exclusively occupy the top 10 terms in this visualization.

The overall it's clear Dexamethasone actively suppresses cell proliferation while strongly inducing a distinct metabolic state.

## **Graded Exercises: (11 marks)**

**Note on Biology:** For the following questions, it's helpful to know that Albuterol is a beta-2 agonist. Its primary function in asthma treatment is to relax the airway smooth muscles (bronchodilation), which is a different mechanism from the anti-inflammatory action of Dexamethasone.

**GQ*1: Prepare and run GSEA for the `'Albuterol'` treatment group. Using the `dge_df` DataFrame, create a ranked gene list for the `'Albuterol'` comparison against the `'Untreated'` control and run GSEA using the `'MSigDB_Hallmark_2020'` gene set library. (3 marks)**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Write your code here

# Display the head of the results to confirm it ran
gsea_results_alb.res2d.head()

**GQ*2: Visualize the GSEA results for the `'Albuterol'` treatment and interpret them. Create a bar plot showing the significant pathways (`FDR q-val < 0.25`). Based on the plot and the known function of `'Albuterol'`, what biological story do these results suggest? How does it differ from the `'Dexamethasone'` results? (4 marks)**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Write your code here


<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

**GQ*3: Explore a different gene set library. The Gene Ontology (GO) library provides a more granular view of biology. Rerun the GSEA for the Albuterol_Dexamethasone treatment, but this time use the `'KEGG_2021_Human'` library. Display the top 10 up- and down-regulated terms. What new insights, if any, do you gain from this more detailed analysis? (4 marks)**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Write your code here


<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Conclusion

In this module, we moved beyond single-gene analysis and embraced a systems-level view with Gene Set Enrichment Analysis. By leveraging our DGE results from last week, we successfully identified the major biological pathways that are collectively altered by Dexamethasone treatment.

You have learned how to:
- Prepare a ranked gene list, the fundamental input for GSEA.
- Use `gseapy` to test for enrichment against the MSigDB Hallmark gene sets.
- Interpret enrichment plots, NES, and FDR q-values to understand the results.
- Create summary visualizations that tell a clear biological story, connecting the computational analysis back to the known functions of the drug.

GSEA is a big part of modern transcriptomics analysis. It transforms long lists of genes into actionable biological insights, providing a powerful framework for forming new hypotheses. You are now equipped to apply this technique to your own data and uncover the hidden biological themes within.